In [1]:
import os
print(os.listdir('/kaggle/input/'))

['datasets']


In [2]:
for root, dirs, files in os.walk('/kaggle/input'):
    print(root)
    print(files)

/kaggle/input
[]
/kaggle/input/datasets
[]
/kaggle/input/datasets/ivanovanastyaaa
[]
/kaggle/input/datasets/ivanovanastyaaa/aaaaaaaaaaaaaaaa
['rtdetr_l_best.pt']
/kaggle/input/datasets/ivanovanastyaaa/ivanova-pril
['yolov9c_best.pt', 'efficientdet_best.pt', 'yolov8n_best.pt', 'ssd_best.pt', 'rtdetr_l_best.pt']
/kaggle/input/datasets/ivanovanastyaaa/my-5-bert-weights-ivanova
['yolov9c_best.pt', 'efficientdet_best.pt', 'yolov8n_best.pt', 'ssd_best.pt', 'rtdetr_l_best.pt']
/kaggle/input/datasets/ivanovanastyaaa/weights-for-work
['yolov9c_best.pt', 'efficientdet_best.pt', 'yolov8n_best.pt', 'ssd_best.pt', 'rtdetr_l_best.pt']


In [3]:
!pip install ultralytics gradio -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 34.1 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.2/53.2 kB 2.8 MB/s eta 0:00:00


In [4]:
from ultralytics import RTDETR
import gradio as gr
from PIL import Image

MODEL_PATH = "/kaggle/input/datasets/ivanovanastyaaa/weights-for-work/rtdetr_l_best.pt"

model = RTDETR(MODEL_PATH)

def detect(image):
    results = model.predict(source=image, conf=0.25)

    plotted = Image.fromarray(results[0].plot())

    detections = []
    boxes = results[0].boxes

    for box in boxes:
        cls = int(box.cls[0])
        conf = float(box.conf[0])

        detections.append(
            f"{model.names[cls]} : {conf:.2f}"
        )

    return plotted, "\n".join(detections)

app = gr.Interface(
    fn=detect,
    inputs=gr.Image(type="pil"),
    outputs=[
        gr.Image(label="Результат"),
        gr.Textbox(label="Обнаруженные объекты")
    ],
    title="RT-DETR Detector"
)

app.launch(debug=True)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
* Running on local URL:  http://127.0.0.1:7860
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

* Running on public URL: https://411e28c2f7571277e6.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://411e28c2f7571277e6.gradio.live


In [8]:
!pip install ultralytics gradio openpyxl -q

In [9]:
from ultralytics import RTDETR
import gradio as gr
from PIL import Image
import json
import pandas as pd
import os
from datetime import datetime

# путь к модели
MODEL_PATH = "/kaggle/input/datasets/ivanovanastyaaa/weights-for-work/rtdetr_l_best.pt"

# загрузка модели
model = RTDETR(MODEL_PATH)

# файл истории
history_file = "/kaggle/working/history.json"

# если файла нет — создать пустой
if not os.path.exists(history_file):
    with open(history_file, "w", encoding="utf-8") as f:
        json.dump([], f, ensure_ascii=False)


# функция детекции
def detect(image):

    results = model.predict(
        source=image,
        conf=0.25
    )

    result_img = Image.fromarray(
        results[0].plot()
    )

    boxes = results[0].boxes

    classes = []
    confidences = []

    for box in boxes:

        cls = int(box.cls[0])
        conf = round(float(box.conf[0]),3)

        classes.append(
            model.names[cls]
        )

        confidences.append(
            conf
        )

    # чтение истории
    with open(history_file, encoding='utf-8') as f:
        history = json.load(f)

    # запись результата
    record = {

        'время':
        datetime.now().strftime("%Y-%m-%d %H:%M:%S"),

        'модель':
        'RT-DETR',

        'num_defects':
        len(classes),

        'классы':
        classes,

        'уверенность':
        confidences
    }

    history.append(record)

    # сохранение истории
    with open(history_file, 'w', encoding='utf-8') as f:

        json.dump(
            history,
            f,
            ensure_ascii=False,
            indent=4
        )

    text = "\n".join(
        [f"{c}: {p}" for c,p in zip(classes,confidences)]
    )

    if len(text)==0:
        text="Объекты не обнаружены"

    return result_img,text


# создание Excel отчёта
def create_report():

    with open(history_file, encoding='utf-8') as f:
        history = json.load(f)

    rows=[]

    for item in history:

        rows.append({

            'Время':
            item['время'],

            'Модель':
            item['модель'],

            'Найдено дефектов':
            item['num_defects'],

            'Классы':
            ', '.join(item['классы'])
            if item['классы']
            else 'нет',

            'Уверенность':
            ', '.join(
                map(str,item['уверенность'])
            )
            if item['уверенность']
            else 'нет'
        })

    df = pd.DataFrame(rows)

    report_path="/kaggle/working/report.xlsx"

    df.to_excel(
        report_path,
        index=False
    )

    return report_path


# интерфейс Gradio
with gr.Blocks() as demo:

    gr.Markdown("# Детекция дефектов RT-DETR")

    input_img = gr.Image(
        type="pil",
        label="Загрузите изображение"
    )

    output_img = gr.Image(
        label="Результат"
    )

    result_text = gr.Textbox(
        label="Найденные объекты"
    )

    detect_btn = gr.Button(
        "Запустить детекцию"
    )

    report_btn = gr.Button(
        "Создать Excel отчёт"
    )

    report_file = gr.File(
        label="Скачать отчёт"
    )

    detect_btn.click(
        fn=detect,
        inputs=input_img,
        outputs=[
            output_img,
            result_text
        ]
    )

    report_btn.click(
        fn=create_report,
        outputs=report_file
    )


demo.launch(share=True)

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://67788a836ff51ba694.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)



0: 640x640 2 D00s, 2 D10s, 1 D20, 1975.1ms
Speed: 33.4ms preprocess, 1975.1ms inference, 3.6ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 2 D00s, 2 D10s, 1355.3ms
Speed: 2.6ms preprocess, 1355.3ms inference, 0.6ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 1 D20, 1335.9ms
Speed: 2.9ms preprocess, 1335.9ms inference, 0.7ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 1 D00, 4 D20s, 1304.4ms
Speed: 3.0ms preprocess, 1304.4ms inference, 0.6ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 3 D00s, 2 D20s, 1357.1ms
Speed: 3.3ms preprocess, 1357.1ms inference, 0.5ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 6 D00s, 2 D20s, 1312.9ms
Speed: 3.6ms preprocess, 1312.9ms inference, 0.8ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 3 D20s, 1359.3ms
Speed: 2.8ms preprocess, 1359.3ms inference, 0.7ms postprocess per image at shape (1, 3, 640, 640)
